In [0]:
dbutils.widgets.text("catalogo", "electrocasa_prod", "Catálogo del proyecto")
catalogo = dbutils.widgets.get("catalogo")
print(f"Configurando catálogo: {catalogo}")

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS IDENTIFIER(:catalogo);
USE CATALOG IDENTIFIER(:catalogo);

CREATE SCHEMA IF NOT EXISTS bronze;
CREATE SCHEMA IF NOT EXISTS silver;
CREATE SCHEMA IF NOT EXISTS gold;

CREATE VOLUME IF NOT EXISTS default.vol_landing;

In [0]:
fuentes = ["ventas", "catalogo", "empleados", "resenas", "devoluciones", "sucursales"]
base_path = f"/Volumes/{catalogo}/default/vol_landing"

for f in fuentes:
    ruta = f"{base_path}/{f}"
    dbutils.fs.mkdirs(ruta)
    print(f"Carpeta creada: {ruta}")

In [0]:
%sql
USE CATALOG IDENTIFIER(:catalogo);

CREATE OR REPLACE FUNCTION default.mask_dni(val STRING)
RETURNS STRING
RETURN CASE 
  WHEN is_account_group_member('electrocasa_ingenieria') THEN val
  ELSE '***'
END;

CREATE OR REPLACE FUNCTION default.mask_salario(val DECIMAL(12,2))
RETURNS DECIMAL(12,2)
RETURN CASE 
  WHEN is_account_group_member('electrocasa_ingenieria') THEN val
  ELSE NULL
END;

In [0]:
%sql
USE CATALOG IDENTIFIER(:catalogo);

-- Ingeniería: lectura y escritura en bronze, silver y gold
GRANT USE CATALOG ON CATALOG IDENTIFIER(:catalogo) TO `electrocasa_ingenieria`;
GRANT ALL PRIVILEGES ON SCHEMA bronze TO `electrocasa_ingenieria`;
GRANT ALL PRIVILEGES ON SCHEMA silver TO `electrocasa_ingenieria`;
GRANT ALL PRIVILEGES ON SCHEMA gold TO `electrocasa_ingenieria`;
GRANT READ VOLUME, WRITE VOLUME ON VOLUME default.vol_landing TO `electrocasa_ingenieria`;

-- Analistas: solo lectura en gold
GRANT USE CATALOG ON CATALOG IDENTIFIER(:catalogo) TO `electrocasa_analistas`;
GRANT USE SCHEMA ON SCHEMA gold TO `electrocasa_analistas`;
GRANT SELECT ON SCHEMA gold TO `electrocasa_analistas`;

-- Auditoría: solo lectura en gold
GRANT USE CATALOG ON CATALOG IDENTIFIER(:catalogo) TO `electrocasa_auditoria`;
GRANT USE SCHEMA ON SCHEMA gold TO `electrocasa_auditoria`;
GRANT SELECT ON SCHEMA gold TO `electrocasa_auditoria`;

In [0]:
%sql
USE CATALOG IDENTIFIER(:catalogo);

-- 1. Demostración de REVOKE (toma captura del antes y después)
GRANT SELECT ON SCHEMA silver TO `electrocasa_analistas`;
SHOW GRANTS ON SCHEMA silver;

REVOKE SELECT ON SCHEMA silver FROM `electrocasa_analistas`;
SHOW GRANTS ON SCHEMA silver;

-- 2. Permisos de Auditoría sobre system

In [0]:
%sql
GRANT USE CATALOG ON CATALOG system TO `electrocasa_auditoria`;
GRANT USE SCHEMA ON SCHEMA system.access TO `electrocasa_auditoria`;
GRANT SELECT ON TABLE system.access.audit TO `electrocasa_auditoria`;
GRANT SELECT ON TABLE system.access.table_lineage TO `electrocasa_auditoria`;
GRANT SELECT ON TABLE system.access.column_lineage TO `electrocasa_auditoria`;